# Llama-3.2-1B QLoRA


## Setup

In [ ]:
!pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1
!pip install -U "trl[peft]" transformers peft accelerate bitsandbytes

  Using cached bitsandbytes-0.50.1-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached trl-1.10.0-py3-none-any.whl.metadata (12 kB)
Using cached bitsandbytes-0.50.1-py3-none-manylinux_2_24_x86_64.whl (41.0 MB)
Using cached trl-1.10.0-py3-none-any.whl (925 kB)
  Attempting uninstall: bitsandbytes
    Found existing installation: bitsandbytes 0.48.2
    Uninstalling bitsandbytes-0.48.2:
      Successfully uninstalled bitsandbytes-0.48.2
  Attempting uninstall: trl
    Found existing installation: trl 0.25.1
    Uninstalling trl-0.25.1:
      Successfully uninstalled trl-0.25.1


In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import re
import math
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, set_seed, BitsAndBytesConfig
from datasets import load_dataset, Dataset, DatasetDict
import wandb
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datetime import datetime
import matplotlib.pyplot as plt

## Hyperparameters



In [ ]:
#hyperparameters

EPOCHS = 3


EFFECTIVE_BATCH = 32
BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 4
GRAD_CHECKPOINTING = False

MAX_SEQUENCE_LENGTH = 128

In [ ]:
#hyperparameters for qlora
QUANT_4_BIT = True
LORA_R = 32
LORA_ALPHA = LORA_R * 2
ATTENTION_LAYERS = ["q_proj", "v_proj", "k_proj", "o_proj"]
MLP_LAYERS = ["gate_proj", "up_proj", "down_proj"]
LORA_DROPOUT = 0.1
TARGET_MODULES = ATTENTION_LAYERS

In [ ]:

LEARNING_RATE = 2e-4


WARMUP = 0.03
LR_SCHEDULER_TYPE = 'cosine'
WEIGHT_DECAY = 0.001


OPTIMIZER = "adamw_torch"


capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8
print("GPU compute capability:", capability, "| bf16-capable:", use_bf16)

GPU compute capability: (7, 5) | bf16-capable: False


In [ ]:
#for tracking
LOG_STEPS = 5


SAVE_STEPS = 88
LOG_TO_WANDB = True

In [ ]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token, add_to_git_credential=True)

## Load dataset

In [ ]:
from datasets import load_dataset
DATASET_NAME = "toughfigure/jobs_prompts_full"
dataset = load_dataset(DATASET_NAME)
train = dataset['train']
val = dataset['validation']
test = dataset['test']

print(train)
print(val)
print(test)

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 2797
})
Dataset({
    features: ['prompt', 'completion'],
    num_rows: 155
})
Dataset({
    features: ['prompt', 'completion'],
    num_rows: 155
})


In [ ]:
import re

def strip_answer(example):
    example["prompt"] = re.sub(r"\n*Salary is \$[\d.,]+\s*$", "", example["prompt"]).strip()
    return example

train = train.map(strip_answer)
val = val.map(strip_answer)
test = test.map(strip_answer)

# sanity check — prompt should no longer end with the answer
print(repr(train[0]["prompt"][-80:]))
print(train[0]["completion"])

'gram to receive monetary or non-monetary recognition awards. Other incentives ma'
Salary is $170940.00


## Load tokenizer (before the model, so we can measure prompt lengths)

In [ ]:
BASE_MODEL = "meta-llama/Llama-3.2-1B"

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

## Pick a real `MAX_SEQUENCE_LENGTH`

Measure actual tokenized prompt+completion lengths across a sample of the
(now-cleaned) training set, instead of guessing.

In [ ]:
sample = train.select(range(min(500, len(train))))
lengths = [
    len(tokenizer(p + " " + c)["input_ids"])
    for p, c in zip(sample["prompt"], sample["completion"])
]

print("min:", min(lengths), "max:", max(lengths), "mean:", sum(lengths) / len(lengths))

# Pick a length that comfortably covers most examples without being wasteful.
# Adjust the percentile/padding below if your max print above is much higher.
import statistics
p95 = sorted(lengths)[int(0.95 * len(lengths))]
MAX_SEQUENCE_LENGTH = min(1024, int(p95 * 1.1))  # small headroom, capped at 1024
print("Using MAX_SEQUENCE_LENGTH =", MAX_SEQUENCE_LENGTH)

min: 117 max: 620 mean: 514.758
Using MAX_SEQUENCE_LENGTH = 682


## Quantization config



In [ ]:
from transformers import BitsAndBytesConfig

if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
  )
else:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.float16,
  )

## Load the quantized base model

In [ ]:
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.float16,
    quantization_config=quant_config,
    device_map="auto",
    # SPEED: T4 (sm_75) can't run FlashAttention-2, but SDPA's memory-efficient
    # kernel still beats the eager attention path.
    attn_implementation="sdpa",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

# SPEED: no KV cache during training - it allocates memory we never read.
base_model.config.use_cache = False

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Memory footprint: 1012.0 MB


## LoRA config

In [ ]:
from peft import LoraConfig, get_peft_model

lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias='none',
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

peft_model = get_peft_model(base_model, lora_parameters)

for _, param in peft_model.named_parameters():
    if param.requires_grad and param.dtype != torch.float32:
        param.data = param.data.to(torch.float32)

peft_model.print_trainable_parameters()
print("trainable dtypes:", {p.dtype for p in peft_model.parameters() if p.requires_grad})

trainable params: 6,815,744 || all params: 1,242,630,144 || trainable%: 0.5485
trainable dtypes: {torch.float32}


## Run naming

In [ ]:
from datetime import datetime

PROJECT_RUN_NAME = "llama-3.2-qlora"
RUN_NAME = f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
HF_USER = "toughfigure"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

## Training config



In [ ]:
from trl import SFTConfig

train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,

    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    gradient_checkpointing=GRAD_CHECKPOINTING,
    gradient_checkpointing_kwargs={"use_reentrant": False} if GRAD_CHECKPOINTING else None,

    per_device_eval_batch_size=max(BATCH_SIZE, 8),

    optim=OPTIMIZER,


    train_sampling_strategy="group_by_length",
    dataloader_num_workers=2,
    dataloader_pin_memory=True,

    save_steps=SAVE_STEPS,
    save_total_limit=2,
    logging_steps=LOG_STEPS,

    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP,
    fp16=True,
    bf16=False,

    max_grad_norm=0.3,
    max_steps=-1,

    lr_scheduler_type=LR_SCHEDULER_TYPE,

    report_to="wandb" if LOG_TO_WANDB else None,
    run_name=RUN_NAME,

    max_length=MAX_SEQUENCE_LENGTH,
    save_strategy="steps",

    push_to_hub=False,

    eval_strategy="steps",
    eval_steps=SAVE_STEPS,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

print(f"micro-batch {BATCH_SIZE} x accum {GRADIENT_ACCUMULATION_STEPS} "
      f"= effective {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")

micro-batch 4 x accum 8 = effective 32


## Build the trainer


In [ ]:
from transformers import EarlyStoppingCallback

fine_tuning = SFTTrainer(
    model=peft_model,
    train_dataset=train,
    eval_dataset=val,
    args=train_parameters,

    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

Tokenizing train dataset:   0%|          | 0/2797 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2797 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2797 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/2797 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/155 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/155 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/155 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/155 [00:00<?, ? examples/s]

## Fine-tune

In [ ]:
# Fine-tune!
fine_tuning.train()

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
88,0.564991,0.724186,0.677900,1458897.000000,0.848642
176,0.643371,0.675495,0.661802,2897921.000000,0.850168
264,0.361521,0.654202,0.659507,4336945.000000,0.855613


wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run


TrainOutput(global_step=264, training_loss=0.8564802561745499, metrics={'train_runtime': 1831.2971, 'train_samples_per_second': 4.582, 'train_steps_per_second': 0.144, 'total_flos': 2.556854933070643e+16, 'train_loss': 0.8564802561745499, 'epoch': 3.0})

In [ ]:
tokenizer.padding_side = "left"

In [ ]:
import re
import torch

def extract_salary(text):
    """Pull the numeric salary out of a completion string like 'Salary is $170940.00'."""
    m = re.search(r"\$?([\d,]+(?:\.\d+)?)", text)
    if not m:
        return None
    return float(m.group(1).replace(",", ""))

peft_model.eval()
peft_model.config.use_cache = True  # re-enable for faster generation

preds, actuals = [], []
BATCH = 16

for i in range(0, len(test), BATCH):
    batch_prompts = test["prompt"][i:i+BATCH]
    batch_actuals = test["completion"][i:i+BATCH]

    inputs = tokenizer(
        [p + "\nSalary is $" for p in batch_prompts],
        return_tensors="pt", padding=True, truncation=True,
        max_length=MAX_SEQUENCE_LENGTH
    ).to(peft_model.device)

    with torch.no_grad():
        out = peft_model.generate(
            **inputs, max_new_tokens=16,
            pad_token_id=tokenizer.pad_token_id,
            do_sample=False,
        )

    for j in range(len(batch_prompts)):
        gen_text = tokenizer.decode(out[j][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        pred_salary = extract_salary("$" + gen_text)
        actual_salary = extract_salary(batch_actuals[j])
        if pred_salary is not None and actual_salary is not None:
            preds.append(pred_salary)
            actuals.append(actual_salary)

    print(f"  {i+len(batch_prompts)}/{len(test)} done", end="\r")

peft_model.config.use_cache = False  # restore training setting

# Metrics
import statistics
errors_pct = [abs(p - a) / a * 100 for p, a in zip(preds, actuals) if a != 0]
errors_abs = [abs(p - a) for p, a in zip(preds, actuals)]

print(f"\n\nparsed {len(preds)}/{len(test)} test examples")
print(f"MAPE: {statistics.mean(errors_pct):.1f}%")
print(f"Median absolute error: ${statistics.median(errors_abs):,.0f}")
print(f"Median % error: {statistics.median(errors_pct):.1f}%")

# quick look at a few examples
for p, a in list(zip(preds, actuals))[:5]:
    print(f"  predicted ${p:,.0f}  |  actual ${a:,.0f}  |  off by {abs(p-a)/a*100:.1f}%")

  155/155 done

parsed 155/155 test examples
MAPE: 13.9%
Median absolute error: $7,150
Median % error: 5.6%
  predicted $102,500  |  actual $81,864  |  off by 25.2%
  predicted $112,500  |  actual $112,500  |  off by 0.0%
  predicted $112,500  |  actual $112,500  |  off by 0.0%
  predicted $92,500  |  actual $92,500  |  off by 0.0%
  predicted $100,000  |  actual $142,500  |  off by 29.8%


## Push to hub

In [ ]:
# Push the final (best) adapter to Hugging Face - once, at the end.
# During training push_to_hub was False, so nothing was uploaded mid-run.
fine_tuning.model.push_to_hub(HUB_MODEL_NAME, private=True)
tokenizer.push_to_hub(HUB_MODEL_NAME, private=True)
print(f"Saved to the hub: {HUB_MODEL_NAME}")

README.md:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   2%|2         |  565kB / 27.3MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpn8537018/tokenizer.json:  12%|#2        | 2.13MB / 17.2MB            

Saved to the hub: toughfigure/llama-3.2-qlora
